# Getting Started

Purpose: Illustrate the use of IBM's [terratorch](https://ibm.github.io/terratorch/) library.  

See [quickstart](https://ibm.github.io/terratorch/quick_start/#python).

## What is [Terratorch?](https://ibm.github.io/terratorch/)

Overview

The purpose of this package is to build a flexible fine-tuning framework for Geospatial Foundation Models (GFMs) based on TorchGeo and Lightning which can be employed at different abstraction levels. It currently supports models from the [Prithvi](https://huggingface.co/ibm-nasa-geospatial) and [Granite](https://huggingface.co/ibm-granite/granite-geospatial-land-surface-temperature) series, and also have been tested with others models available on HuggingFace.

This library provides:

* All the functionality in TorchGeo.
* Easy access to Prithvi, timm and smp backbones.
* Flexible trainers for Image Segmentation, Pixel Wise Regression and Classification (more in progress).
* Launching of fine-tuning tasks through powerful configuration files.
* A good starting place is familiarization with [PyTorch Lightning](https://lightning.ai/docs/pytorch/stable/), which this project is built on. [TorchGeo](https://torchgeo.readthedocs.io/en/stable/) is also an important complementary reference.

Check out the [architecture overview](https://ibm.github.io/terratorch/architecture/) for a general description about how TerraTorch is organized.

## Terms

Here are some [terms](https://ibm.github.io/terratorch/glossary/) defined by terratorch that are good to be familiar with.

Encoder

The neural network used to map between the inputs and the intermdiary stage (usually referred as embedding or sometimes as latent space) of the forward step. The encoder is also frequently called backbone and, for finetuning tasks, it is usually the part of the model which is not updated/trained.

Decoder

The neural network employed to map between the intermediary stage (embedding/latent space) and the target output. For finetuning tasks, the decoder is the most essential part, since it is trained to map the embedding produced by a previoulsy trained encoder to a new task.

Head

A network, usually very small when compared to the encoder and decoder, which is used as final step to adapt the decoder output to a specific task, for example, by applying a determined activation to it.

Neck

Necks are operations placed between the encoder and the decoder stages aimed at adjusting possible discrepancies, as incompatible shapes, or applying some specific transform, as a normalization required for the task being executed.

Factory

A Factory is a class which organizes the instantiation of a complete model, as a backbone-neck-decoder-head architecture. A class is intended to receive lists and dictionarie containing the required arguments used to build the model and returns a new instance already ready to be used.


**Note that decoders are not always part of fine-tuning foundation models (just encoder + head) so this terminology is different than in some circles.**

**Also, the encoder often IS updated during finetuning and not held fixed, but it can be.**


In [1]:
import terratorch
from torchsummary import summary

INFO:terratorch:granitewxc not installed, please use pip install granitewxc


## Creating Backbones

In [2]:
from terratorch import BACKBONE_REGISTRY

Internally, terratorch maintains several registries for components such as backbones or decoders. The top-level BACKBONE_REGISTRY collects all of them.

The name passed to build is used to find the appropriate model constructor, which will be the first model from the first registry found with that name.

To explicitly determine the registry that will build the model, you may prepend a prefix such as timm_ to the model name. In this case, the timm model registry will be exclusively searched for the model.

In [3]:
# List all models
list(BACKBONE_REGISTRY)[:10] @

['terratorch_prithvi_eo_tiny',
 'terratorch_prithvi_eo_v1_100',
 'terratorch_prithvi_eo_v2_300',
 'terratorch_prithvi_eo_v2_600',
 'terratorch_prithvi_eo_v2_300_tl',
 'terratorch_prithvi_eo_v2_600_tl',
 'terratorch_prithvi_vit_tiny',
 'terratorch_prithvi_vit_100',
 'terratorch_timm_prithvi_eo_v1_100',
 'terratorch_timm_prithvi_eo_v2_300']

In [4]:
# Find available prithvi models
print([model_name for model_name in BACKBONE_REGISTRY if "terratorch_prithvi" in model_name])

['terratorch_prithvi_eo_tiny', 'terratorch_prithvi_eo_v1_100', 'terratorch_prithvi_eo_v2_300', 'terratorch_prithvi_eo_v2_600', 'terratorch_prithvi_eo_v2_300_tl', 'terratorch_prithvi_eo_v2_600_tl', 'terratorch_prithvi_vit_tiny', 'terratorch_prithvi_vit_100']


In [70]:
# The backbone registry prefix (e.g. `terratorch` or `timm`) is optional - in this case, the underlying registry is terratorch.
model = BACKBONE_REGISTRY.build(
    "prithvi_eo_v2_300", 
    pretrained=True,
    num_frames=1, # Number of frames associated with each chip (t=1, h, w)
    # ckpt_path='path/to/model.pt' # Instead of pretrained, can load your own weights
)

INFO:terratorch.models.backbones.prithvi_vit:Model bands not passed. Assuming bands are ordered in the same way as [<HLSBands.BLUE: 'BLUE'>, <HLSBands.GREEN: 'GREEN'>, <HLSBands.RED: 'RED'>, <HLSBands.NIR_NARROW: 'NIR_NARROW'>, <HLSBands.SWIR_1: 'SWIR_1'>, <HLSBands.SWIR_2: 'SWIR_2'>].Pretrained patch_embed layer may be misaligned with current bands
INFO:root:Loaded weights for HLSBands.BLUE in position 0 of patch embed
INFO:root:Loaded weights for HLSBands.GREEN in position 1 of patch embed
INFO:root:Loaded weights for HLSBands.RED in position 2 of patch embed
INFO:root:Loaded weights for HLSBands.NIR_NARROW in position 3 of patch embed
INFO:root:Loaded weights for HLSBands.SWIR_1 in position 4 of patch embed
INFO:root:Loaded weights for HLSBands.SWIR_2 in position 5 of patch embed


In [51]:
model.in_chans, model.img_size

(6, (224, 224))

In [52]:
# The model creates patches that are 16x16 with 1024 channels initially
model.patch_embed

PatchEmbed(
  (proj): Conv3d(6, 1024, kernel_size=(1, 16, 16), stride=(1, 16, 16))
  (norm): Identity()
)

In [53]:
# Thus, we end up with a grid of 14x14 patches
224/16

14.0

In [54]:
# These are flattened
14*14

196

In [22]:
summary(model, (6, 224, 224))

Layer (type:depth-idx)                   Output Shape              Param #
├─PatchEmbed: 1-1                        [-1, 196, 1024]           --
|    └─Conv3d: 2-1                       [-1, 1024, 1, 14, 14]     1,573,888
|    └─Identity: 2-2                     [-1, 196, 1024]           --
├─ModuleList: 1                          []                        --
|    └─Block: 2-3                        [-1, 197, 1024]           --
|    |    └─LayerNorm: 3-1               [-1, 197, 1024]           2,048
|    |    └─Attention: 3-2               [-1, 197, 1024]           4,198,400
|    |    └─Identity: 3-3                [-1, 197, 1024]           --
|    |    └─Identity: 3-4                [-1, 197, 1024]           --
|    |    └─LayerNorm: 3-5               [-1, 197, 1024]           2,048
|    |    └─Mlp: 3-6                     [-1, 197, 1024]           8,393,728
|    |    └─Identity: 3-7                [-1, 197, 1024]           --
|    |    └─Identity: 3-8                [-1, 197, 1024]  

Layer (type:depth-idx)                   Output Shape              Param #
├─PatchEmbed: 1-1                        [-1, 196, 1024]           --
|    └─Conv3d: 2-1                       [-1, 1024, 1, 14, 14]     1,573,888
|    └─Identity: 2-2                     [-1, 196, 1024]           --
├─ModuleList: 1                          []                        --
|    └─Block: 2-3                        [-1, 197, 1024]           --
|    |    └─LayerNorm: 3-1               [-1, 197, 1024]           2,048
|    |    └─Attention: 3-2               [-1, 197, 1024]           4,198,400
|    |    └─Identity: 3-3                [-1, 197, 1024]           --
|    |    └─Identity: 3-4                [-1, 197, 1024]           --
|    |    └─LayerNorm: 3-5               [-1, 197, 1024]           2,048
|    |    └─Mlp: 3-6                     [-1, 197, 1024]           8,393,728
|    |    └─Identity: 3-7                [-1, 197, 1024]           --
|    |    └─Identity: 3-8                [-1, 197, 1024]  

In [55]:
import torch
import numpy as np

X = np.random.random((1, 6, 224, 224))
out = model(torch.Tensor(X))

In [57]:
# It appears they expose the attention scores from each block
len(out)

24

In [58]:
out[-1].shape

torch.Size([1, 197, 1024])

## Directly creating a full model

In [59]:
from terratorch.models import EncoderDecoderFactory
from terratorch.datasets import HLSBands

# Models constructed by the EncoderDecoderFactory have an internal structure explicitly divided into backbones, necks, decoders and heads. 
# This structure is provided by the PixelWiseModel and ScalarOutputModel classes.
model_factory = EncoderDecoderFactory()

# Let's build a segmentation model
# Parameters prefixed with backbone_ get passed to the backbone
# Parameters prefixed with decoder_ get passed to the decoder
# Parameters prefixed with head_ get passed to the head

In [60]:
?model_factory.build_model

Signature:
model_factory.build_model(
    task: str,
    backbone: str | torch.nn.modules.module.Module,
    decoder: str | torch.nn.modules.module.Module,
    num_classes: int | None = None,
    necks: list[dict] | None = None,
    aux_decoders: list[terratorch.models.model.AuxiliaryHead] | None = None,
    rescale: bool = True,
    peft_config: dict | None = None,
    **kwargs,
) -> terratorch.models.model.Model
Docstring:
Generic model factory that combines an encoder and decoder, together with a head, for a specific task.

Further arguments to be passed to the backbone, decoder or head. They should be prefixed with
`backbone_`, `decoder_` and `head_` respectively.

Args:
    task (str): Task to be performed. Currently supports "segmentation", "regression" and "classification".
    backbone (str, nn.Module): Backbone to be used. If a string, will look for such models in the different
        registries supported (internal terratorch registry, timm, ...). If a torch nn.Module, will u

Documentation on [tasks](https://ibm.github.io/terratorch/tasks/).

Documentation on available [decoders](https://ibm.github.io/terratorch/decoders/).

In [72]:
model = model_factory.build_model(
    task="segmentation",
    backbone="prithvi_eo_v2_300",
    backbone_pretrained=True,
    backbone_bands=[ # Best to specify this so it is not assumed - the INFO outputs below point you to this ordering
        HLSBands.BLUE,
        HLSBands.GREEN,
        HLSBands.RED,
        HLSBands.NIR_NARROW,
        HLSBands.SWIR_1,
        HLSBands.SWIR_2,
    ],
    necks=[{"name": "SelectIndices", "indices": [-1]}, # Select indices and reshape from "arrays" to "images" that can be used by CNN decoder
           {"name": "ReshapeTokensToImage"}],
    decoder="FCNDecoder",
    decoder_channels=128, # You can choose this
    head_dropout=0.1,
    num_classes=4,
)

# Rest of your PyTorch / PyTorchLightning code

INFO:root:Loaded weights for HLSBands.BLUE in position 0 of patch embed
INFO:root:Loaded weights for HLSBands.GREEN in position 1 of patch embed
INFO:root:Loaded weights for HLSBands.RED in position 2 of patch embed
INFO:root:Loaded weights for HLSBands.NIR_NARROW in position 3 of patch embed
INFO:root:Loaded weights for HLSBands.SWIR_1 in position 4 of patch embed
INFO:root:Loaded weights for HLSBands.SWIR_2 in position 5 of patch embed


In [73]:
summary(model, (6, 224, 224))

Layer (type:depth-idx)                        Output Shape              Param #
├─PrithviViT: 1-1                             [-1, 197, 1024]           --
|    └─PatchEmbed: 2-1                        [-1, 196, 1024]           --
|    |    └─Conv3d: 3-1                       [-1, 1024, 1, 14, 14]     1,573,888
|    |    └─Identity: 3-2                     [-1, 196, 1024]           --
|    └─ModuleList: 2                          []                        --
|    |    └─Block: 3-3                        [-1, 197, 1024]           12,596,224
|    |    └─Block: 3-4                        [-1, 197, 1024]           12,596,224
|    |    └─Block: 3-5                        [-1, 197, 1024]           12,596,224
|    |    └─Block: 3-6                        [-1, 197, 1024]           12,596,224
|    |    └─Block: 3-7                        [-1, 197, 1024]           12,596,224
|    |    └─Block: 3-8                        [-1, 197, 1024]           12,596,224
|    |    └─Block: 3-9                  

Layer (type:depth-idx)                        Output Shape              Param #
├─PrithviViT: 1-1                             [-1, 197, 1024]           --
|    └─PatchEmbed: 2-1                        [-1, 196, 1024]           --
|    |    └─Conv3d: 3-1                       [-1, 1024, 1, 14, 14]     1,573,888
|    |    └─Identity: 3-2                     [-1, 196, 1024]           --
|    └─ModuleList: 2                          []                        --
|    |    └─Block: 3-3                        [-1, 197, 1024]           12,596,224
|    |    └─Block: 3-4                        [-1, 197, 1024]           12,596,224
|    |    └─Block: 3-5                        [-1, 197, 1024]           12,596,224
|    |    └─Block: 3-6                        [-1, 197, 1024]           12,596,224
|    |    └─Block: 3-7                        [-1, 197, 1024]           12,596,224
|    |    └─Block: 3-8                        [-1, 197, 1024]           12,596,224
|    |    └─Block: 3-9                  

In [69]:
model.patch_size

[1, 16, 16]

## Training with Lightning Tasks

In [ ]:
model_args = dict(
  backbone="prithvi_eo_v2_300",
  backbone_pretrained=True,
  backbone_num_frames=1,
  backbone_bands=[
      HLSBands.BLUE,
      HLSBands.GREEN,
      HLSBands.RED,
      HLSBands.NIR_NARROW,
      HLSBands.SWIR_1,
      HLSBands.SWIR_2,
  ],
  necks=[{"name": "SelectIndices", "indices": [-1]},
               {"name": "ReshapeTokensToImage"}],
  decoder="FCNDecoder",
  decoder_channels=128,
  head_dropout=0.1
)

task = PixelwiseRegressionTask(
    model_args,
    "EncoderDecoderFactory",
    loss="rmse",
    lr=lr,
    ignore_index=-1,
    optimizer="AdamW",
    optimizer_hparams={"weight_decay": 0.05},
)

# Pass this LightningModule to a Lightning Trainer, together with some LightningDataModule